# Sillon YOLO v2.0 — Détecteur temps réel sur viseur caméra

Entraîne un détecteur léger `YOLO11n` (export TFLite INT8) sur :
- **PlantDoc Object Detection** (`yusufmurtaza01/plantdoc-object-detection-dataset`, `CC BY 4.0`)
- **PlantSeg** (`weitianqi/plantseg`, autorisation écrite du 9 nov. 2026 : conversion des masques potagers en boîtes englobantes)

Pré-requis Kaggle : **Accelerator GPU T4**, **Internet On** (pour installer `ultralytics`).

In [ ]:
PLANTDOC_OD_HINT = "plantdoc-object-detection-dataset"
PLANTSEG_HINT = "plantseg"
SCRIPTS_HINT = "sillon-ml-scripts"
EPOCHS = 80
IMGSZ = 512

In [ ]:
import subprocess, sys
from pathlib import Path

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "onnx", "onnx2tf", "tf_keras"])

INPUT = Path("/kaggle/input")
def find_input(hint):
    dirs = []
    for p in INPUT.iterdir():
        if p.name == "datasets" and p.is_dir():
            for owner in p.iterdir():
                if owner.is_dir():
                    dirs.extend(d for d in owner.iterdir() if d.is_dir())
        elif p.is_dir():
            dirs.append(p)
    return next((d for d in sorted(dirs) if hint.lower() in d.name.lower()), None)

scripts_dir = find_input(SCRIPTS_HINT)
pdoc_dir = find_input(PLANTDOC_OD_HINT)
pseg_dir = find_input(PLANTSEG_HINT)
assert scripts_dir and pdoc_dir and pseg_dir, f"Manquant: scripts={scripts_dir}, pdoc={pdoc_dir}, pseg={pseg_dir}"

cmd = [
    sys.executable, str(scripts_dir / "train_sillon_yolo.py"),
    "--plantdoc-od", str(pdoc_dir),
    "--plantseg", str(pseg_dir),
    "--out", "/kaggle/working/out_yolo",
    "--epochs", str(EPOCHS),
    "--imgsz", str(IMGSZ),
]
print("$", " ".join(cmd))
subprocess.check_call(cmd)